# 01 - Data Validation

This notebook performs the initial validation of the raw dataset. The objective is to verify that the dataset exists, can be loaded successfully, and has the expected dimensions before any preprocessing begins.

In [1]:
from pathlib import Path
import pandas as pd

## Load Dataset

Load the raw dataset from the project's data directory and verify that the file is available.

In [2]:
DATA_PATH = Path("../data/raw/creditcard.csv")

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")

Dataset loaded successfully.


## Basic File Validation

Verify the dataset's shape and confirm that it matches the expected structure.

In [3]:
EXPECTED_ROWS = 284807
EXPECTED_COLUMNS = 31

print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

assert df.shape[0] == EXPECTED_ROWS, "Unexpected number of rows."
assert df.shape[1] == EXPECTED_COLUMNS, "Unexpected number of columns."

print("File validation passed.")

Rows    : 284807
Columns : 31
File validation passed.


## Schema Validation

Validate that the dataset contains all expected columns in the correct order. This ensures the raw data matches the schema required for downstream processing.

In [4]:
expected_columns = [
    "Time",
    *[f"V{i}" for i in range(1, 29)],
    "Amount",
    "Class"
]

actual_columns = list(df.columns)

print("Expected Columns :", len(expected_columns))
print("Actual Columns   :", len(actual_columns))

Expected Columns : 31
Actual Columns   : 31


In [5]:
assert actual_columns == expected_columns, "Schema validation failed."

print("Schema validation passed.")

Schema validation passed.


## Column Information

Review the dataset schema, data types, and memory usage before proceeding with further validation.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     284807 non-nu

## Missing Values Analysis

Analyze the dataset for missing values to ensure data completeness. Missing values can affect model performance and may require preprocessing if present.

In [7]:
missing_values = df.isnull().sum()

missing_summary = (
    missing_values[missing_values > 0]
    .sort_values(ascending=False)
    .to_frame(name="Missing Values")
)

missing_summary

,Missing Values


## Missing Value Statistics

Summarize the overall completeness of the dataset by reporting the total number and percentage of missing values.

In [8]:
total_missing = missing_values.sum()
missing_percentage = (
    total_missing / (df.shape[0] * df.shape[1])
) * 100

print(f"Total Missing Values : {total_missing}")
print(f"Missing Percentage   : {missing_percentage:.4f}%")

Total Missing Values : 0
Missing Percentage   : 0.0000%


In [9]:
if total_missing == 0:
    print("No missing values found.")
else:
    print("Missing values detected.")

No missing values found.


## Duplicate Records Analysis

Check for duplicate transactions in the dataset. Duplicate records can introduce bias during model training and should be identified before data cleaning.

In [10]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate Records : {duplicate_count}")

Duplicate Records : 1081


## Duplicate Record Inspection

If duplicate records are found, preview a sample to understand their nature before deciding whether to remove them.

In [11]:
if duplicate_count > 0:
    display(df[df.duplicated()].head())
else:
    print("No duplicate records found.")

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
33,26.0,-0.529912,0.873892,1.347247,0.145457,0.414209,0.100223,0.711206,0.176066,-0.286717,...,0.046949,0.208105,-0.185548,0.001031,0.098816,-0.552904,-0.073288,0.023307,6.14,0
35,26.0,-0.535388,0.865268,1.351076,0.147575,0.433680,0.086983,0.693039,0.179742,-0.285642,...,0.049526,0.206537,-0.187108,0.000753,0.098117,-0.553471,-0.078306,0.025427,1.77,0
113,74.0,1.038370,0.127486,0.184456,1.109950,0.441699,0.945283,-0.036715,0.350995,0.118950,...,0.102520,0.605089,0.023092,-0.626463,0.479120,-0.166937,0.081247,0.001192,1.18,0
114,74.0,1.038370,0.127486,0.184456,1.109950,0.441699,0.945283,-0.036715,0.350995,0.118950,...,0.102520,0.605089,0.023092,-0.626463,0.479120,-0.166937,0.081247,0.001192,1.18,0
115,74.0,1.038370,0.127486,0.184456,1.109950,0.441699,0.945283,-0.036715,0.350995,0.118950,...,0.102520,0.605089,0.023092,-0.626463,0.479120,-0.166937,0.081247,0.001192,1.18,0


In [12]:
duplicate_percentage = (duplicate_count / len(df)) * 100

print(f"Duplicate Percentage : {duplicate_percentage:.4f}%")

Duplicate Percentage : 0.3796%


## Data Types Validation

Validate the data type of each feature to ensure consistency with the expected schema. Correct data types are essential for reliable preprocessing and model training.

In [13]:
dtype_df = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values
})

dtype_df

,Column,Data Type
0,Time,float64
1,V1,float64
2,V2,float64
3,V3,float64
4,V4,float64
5,V5,float64
6,V6,float64
7,V7,float64
8,V8,float64
9,V9,float64


## Numeric Feature Verification

Verify that all features are numeric. Since this dataset consists of PCA-transformed numerical features along with numerical transaction information, every column should have a numeric data type.

In [14]:
non_numeric_columns = df.select_dtypes(exclude="number").columns.tolist()

print(f"Non-numeric Columns: {len(non_numeric_columns)}")

if non_numeric_columns:
    print(non_numeric_columns)
else:
    print("All columns have numeric data types.")

Non-numeric Columns: 0
All columns have numeric data types.


In [15]:
df.dtypes.value_counts()

float64    30
int64       1
Name: count, dtype: int64

## Target Variable Validation

Validate the target variable to ensure it contains the expected classes and examine the distribution between legitimate and fraudulent transactions. This helps identify any class imbalance before model development.

In [16]:
target_column = "Class"

print(f"Target Column: {target_column}")
print(f"Unique Classes: {sorted(df[target_column].unique())}")

Target Column: Class
Unique Classes: [np.int64(0), np.int64(1)]


## Class Distribution

Analyze the number and percentage of samples in each class. Fraud detection datasets are typically highly imbalanced, making this an important validation step.

In [17]:
class_distribution = (
    df[target_column]
    .value_counts()
    .sort_index()
    .rename(index={0: "Legitimate", 1: "Fraud"})
)

class_distribution

Class
Legitimate    284315
Fraud            492
Name: count, dtype: int64

In [18]:
class_percentage = (
    df[target_column]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .rename(index={0: "Legitimate", 1: "Fraud"})
    .round(4)
)

class_percentage

Class
Legitimate    99.8273
Fraud          0.1727
Name: proportion, dtype: float64

In [19]:
assert set(df[target_column].unique()) == {0, 1}, \
    "Unexpected target classes found."

print("Target variable validation passed.")

Target variable validation passed.
